# Module 03 — Protein Structural Biology
# Notebook 02: Structure Quality Metrics & ML Features

**Series position:** 03/02 — builds on 03/01 (PDB parsing, RMSD, Kabsch)  
**Feeds into:** 06/01 (GNNs use contact maps as adjacency), 07/01 (AF3 quality evaluation)


## TL;DR — Plain English

**What this notebook does:** Takes you from raw 3D coordinates to the quality metrics and ML-ready feature representations that practising structural biologists and ML engineers use every day.

**After this notebook you can:**
- Compute phi/psi backbone dihedral angles from N-CA-C-N atoms using vector cross products and reproduce a Ramachandran plot from scratch
- Assign secondary structure (helix / strand / coil) using a simplified rule-based DSSP approach and summarise composition per protein
- Build a Cα contact map, threshold it at 8 Å, visualise it, and explain why it is the natural input to a graph neural network
- Flatten the upper triangle of a distance matrix into a feature vector and train an SVM to classify helix-rich vs strand-rich proteins
- Interpret B-factors, clash scores, Ramachandran outlier percentages, and MolProbity scores as structure quality indicators
- Implement simple clash detection (atom pairs < 1.5 Å apart) in pure NumPy
- Explain conceptually how cryo-EM resolution and model quality connect to AF3 predictions

**Time:** ~5 hours | **Difficulty:** ★★★★☆ | **Prerequisites:** 03/01 (RMSD, PDB parsing), 00/01 (NumPy), basic trigonometry

**Why it matters for interviews:**  
Isomorphic Labs, Schrödinger, and DeepMind consistently ask about contact maps, Ramachandran plots, and B-factors in ML engineer interviews. This notebook gives you rigorous, from-scratch implementations you can discuss confidently.


## Structure Quality — Concepts for Beginners

| Term | Plain English |
|------|---------------|
| **Ramachandran plot** | Map of backbone angles (phi, psi) for every residue — most residues should fall in allowed regions |
| **phi (phi) angle** | Rotation angle around the N-Calpha bond of a residue backbone |
| **psi (psi) angle** | Rotation angle around the Calpha-C bond of a residue backbone |
| **allowed region** | phi/psi combination that is sterically possible — outside is a steric clash |
| **DSSP** | Algorithm that assigns secondary structure (H=helix, E=sheet, C=coil) from 3D coordinates |
| **contact map** | 2D binary matrix: cell (i,j)=1 if residues i and j are within a distance threshold |
| **pLDDT** | AlphaFold's per-residue confidence score (0-100); >90 = highly confident, <50 = disordered |
| **resolution** | Quality metric for crystal structures in Angstroms; <2.0A = excellent, >3.5A = low resolution |
| **R-factor** | How well the model explains the diffraction data; <0.25 is acceptable |
| **MolProbity** | Web tool that scores structure quality (clashscore, Ramachandran outliers, rotamer outliers) |
| **rotamer** | One of the allowed side-chain conformations for a given amino acid |
| **B-factor normalisation** | Scaling B-factors to mean=0, std=1 to compare across structures with different overall flexibility |

---
## 1. Beginner Frame: What Is Structure Quality, and Why Does It Matter for ML?

### 1.1 What do we mean by 'structure quality'?

When a protein structure is determined experimentally (by X-ray crystallography, cryo-EM, or NMR) or predicted computationally (by AlphaFold 3), the resulting 3D model is never perfect. **Structure quality** answers the question: *how trustworthy are the atomic positions in this model?*

There are two orthogonal axes:

| Axis | What it captures | Key metrics |
|------|-----------------|-------------|
| **Local geometry** | Are bond lengths, angles, and dihedrals chemically sensible? | Ramachandran outliers, clash score, rotamer outliers |
| **Data fit** | Does the model agree with the experimental data? | R-factor (X-ray), map-model CC (cryo-EM), pLDDT (AF3) |

For ML we mostly care about *local geometry* because it tells us whether the backbone trace we are using as input is physically plausible.

### 1.2 Why ML engineers must understand structure quality

1. **Garbage in, garbage out.** Training a GNN on structures with >5 % Ramachandran outliers will teach the model to predict implausible geometries.
2. **B-factors as uncertainty.** High B-factors signal disordered regions — you should down-weight or mask those residues in a loss function.
3. **AF3 confidence vs experimental quality.** AlphaFold 3 reports pLDDT per residue. Knowing how this maps to B-factor and Ramachandran quality helps you set thresholds for downstream tasks.
4. **Contact maps.** These are the canonical ML feature for protein structure (used in AlphaFold 1, 2 distance predictions, and GNN edges in module 06). A poor-quality structure produces a noisy contact map.

### 1.3 Roadmap for this notebook

```
Simulated 3D coordinates
    |
    +---> Dihedral angles (phi, psi)  ---> Ramachandran plot
    |                                          |
    |                                  Secondary structure (DSSP-lite)
    |
    +---> Cα distance matrix  -----------> Contact map (8 Å threshold)
    |          |                                 |
    |          +-> upper-triangle features       +--> GNN adjacency (Module 06)
    |          +-> SVM classifier
    |
    +---> B-factor array  -------> quality mask
    |
    +---> Atom positions  -------> clash detection (<1.5 Å)
```


---
## 2. Ramachandran Plot: Backbone Dihedral Angles

### 2.1 Theory

The backbone of a polypeptide chain is defined by repeating units: **N — Cα — C(=O) — N — Cα — C(=O) — ...**  
The only true degrees of freedom in an ideal peptide bond are the two torsion angles per residue:

- **phi (φ):** rotation around the **N — Cα** bond, defined by atoms **C_{i-1} — N_i — Cα_i — C_i**
- **psi (ψ):** rotation around the **Cα — C** bond, defined by atoms **N_i — Cα_i — C_i — N_{i+1}**

The omega angle (around the peptide C—N bond) is almost always ~180° (trans) and is usually ignored.

### 2.2 Dihedral angle formula

Given four atoms A, B, C, D, the dihedral angle θ is the angle between the plane (A,B,C) and the plane (B,C,D):

```
b1 = B - A
b2 = C - B
b3 = D - C

n1 = b1 × b2      (normal to plane ABC)
n2 = b2 × b3      (normal to plane BCD)

x  = n1 · n2
y  = (b2/|b2| × n1) · n2      (sine component via triple product)

θ  = atan2(y, x)    →  result in [-π, +π]
```

### 2.3 Ramachandran regions (simplified)

| Region | phi (°) range | psi (°) range | Secondary structure |
|--------|--------------|--------------|---------------------|
| Core helix | −90 to −30 | −75 to −5 | α-helix |
| Core strand | −180 to −50 | +90 to +180 | β-strand |
| Left-handed helix | +40 to +90 | +20 to +80 | Rare, Gly only |
| Disallowed | everything else | | Steric clash |

Glycine has no Cβ, so it can access all four quadrants. Proline is constrained by its ring.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

np.random.seed(42)

# ── Helper: dihedral angle from 4 3D points ──────────────────────────────────
def dihedral_angle(a, b, c, d):
    """
    Compute torsion angle defined by four points a, b, c, d (each a 3-vector).
    Returns angle in degrees in [-180, +180].

    Algorithm
    ---------
    1. Build bond vectors b1 = b-a, b2 = c-b, b3 = d-c
    2. n1 = b1 x b2,  n2 = b2 x b3  (plane normals)
    3. Use atan2 so we get the full signed angle in [-π, π]
    """
    b1 = b - a
    b2 = c - b
    b3 = d - c

    # Plane normals
    n1 = np.cross(b1, b2)
    n2 = np.cross(b2, b3)

    # Unit b2 for the sine component
    b2_norm = b2 / (np.linalg.norm(b2) + 1e-12)

    # cos and sin components
    x = np.dot(n1, n2)
    y = np.dot(np.cross(b2_norm, n1), n2)

    return np.degrees(np.arctan2(y, x))


# ── Simulate backbone coordinates ────────────────────────────────────────────
# We simulate 3 protein segments:
#   - alpha_helix:  phi ~ N(-57, 10),  psi ~ N(-47, 10)
#   - beta_strand:  phi ~ N(-119, 10), psi ~ N(+113, 10)
#   - coil/loop:    phi ~ Uniform(-180, 180), psi ~ Uniform(-180, 180)

def simulate_backbone(n_residues=60, segment='helix'):
    """
    Generate synthetic (N, CA, C) backbone coordinates consistent with
    idealized phi/psi angles for the requested secondary structure type.

    Returns
    -------
    coords : np.ndarray shape (n_residues, 3, 3)
        coords[i, 0] = N_i,  coords[i, 1] = CA_i,  coords[i, 2] = C_i
    phi_true : np.ndarray
    psi_true : np.ndarray
    """
    if segment == 'helix':
        phi_true = np.random.normal(-57, 8, n_residues)
        psi_true = np.random.normal(-47, 8, n_residues)
    elif segment == 'strand':
        phi_true = np.random.normal(-119, 10, n_residues)
        psi_true = np.random.normal( 113, 10, n_residues)
    else:  # coil — random but avoid perfect ideal regions
        phi_true = np.random.uniform(-180, 180, n_residues)
        psi_true = np.random.uniform(-180, 180, n_residues)

    # Build idealised backbone using fixed bond lengths (Å) and angles
    # Bond lengths:  N-CA = 1.458, CA-C = 1.525, C-N = 1.329
    # We use these angles as ground truth and reconstruct atom positions
    # via a simplified chain-building approach (no full torsion drive here;
    # we just return the phi/psi values directly and build placeholder coords)

    # For plotting purposes: build simple incremental coords
    coords = np.zeros((n_residues, 3, 3))  # (res, atom, xyz)
    for i in range(n_residues):
        base = np.array([i * 3.8, 0.0, 0.0])  # CA positions along x
        phi_rad = np.radians(phi_true[i])
        psi_rad = np.radians(psi_true[i])
        coords[i, 0] = base + np.array([-1.2, np.sin(phi_rad) * 1.0, np.cos(phi_rad) * 0.5])   # N
        coords[i, 1] = base                                                                       # CA
        coords[i, 2] = base + np.array([ 1.2, np.sin(psi_rad) * 1.0, np.cos(psi_rad) * 0.5])   # C

    return coords, phi_true, psi_true


# Simulate three protein types
n_res = 80
coords_h, phi_h, psi_h = simulate_backbone(n_res, 'helix')
coords_s, phi_s, psi_s = simulate_backbone(n_res, 'strand')
coords_c, phi_c, psi_c = simulate_backbone(n_res, 'coil')

print(f'Helix   phi mean: {phi_h.mean():.1f} +/- {phi_h.std():.1f} deg')
print(f'Strand  phi mean: {phi_s.mean():.1f} +/- {phi_s.std():.1f} deg')
print(f'Coil    phi mean: {phi_c.mean():.1f} +/- {phi_c.std():.1f} deg')

In [ ]:
# ── Ramachandran plot ─────────────────────────────────────────────────────────
# Colour by secondary structure: red = helix, blue = strand, grey = coil

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Left panel: raw scatter coloured by secondary structure ---
ax = axes[0]
ax.scatter(phi_h, psi_h, c='crimson',    alpha=0.5, s=15, label='alpha-helix')
ax.scatter(phi_s, psi_s, c='royalblue',  alpha=0.5, s=15, label='beta-strand')
ax.scatter(phi_c, psi_c, c='grey',       alpha=0.3, s=10, label='coil/loop')

# Mark ideal centres
ax.plot(-57, -47, 'r*', markersize=14, label='alpha ideal (-57,-47)')
ax.plot(-119, 113, 'b*', markersize=14, label='beta ideal (-119,+113)')

# Draw allowed region boundaries (simplified rectangles)
from matplotlib.patches import Rectangle
ax.add_patch(Rectangle((-90, -75), 60, 70, fill=False, edgecolor='crimson',
                        linestyle='--', linewidth=1.5, label='_nolegend_'))
ax.add_patch(Rectangle((-180, 90), 130, 90, fill=False, edgecolor='royalblue',
                        linestyle='--', linewidth=1.5, label='_nolegend_'))

ax.set_xlim(-180, 180)
ax.set_ylim(-180, 180)
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)
ax.set_xlabel('Phi (degrees)', fontsize=12)
ax.set_ylabel('Psi (degrees)', fontsize=12)
ax.set_title('Ramachandran Plot — coloured by secondary structure', fontsize=11)
ax.legend(fontsize=8, loc='upper right')

# Quadrant labels
for (x, y, txt) in [(-130, -130, 'IV'), (60, -130, 'III'),
                     (-130, 130, 'II'),  (60,  130, 'I')]:
    ax.text(x, y, txt, fontsize=9, color='#555555')

# --- Right panel: 2D histogram (density) ---
ax2 = axes[1]
all_phi = np.concatenate([phi_h, phi_s, phi_c])
all_psi = np.concatenate([psi_h, psi_s, psi_c])

h, xedges, yedges, img = ax2.hist2d(
    all_phi, all_psi,
    bins=36, range=[[-180, 180], [-180, 180]],
    cmap='hot_r'
)
plt.colorbar(img, ax=ax2, label='Residue count')
ax2.set_xlabel('Phi (degrees)', fontsize=12)
ax2.set_ylabel('Psi (degrees)', fontsize=12)
ax2.set_title('Ramachandran Density Plot — all residues combined', fontsize=11)
ax2.axhline(0, color='white', linewidth=0.5)
ax2.axvline(0, color='white', linewidth=0.5)

plt.tight_layout()
plt.savefig('/tmp/ramachandran_plot.png', dpi=120, bbox_inches='tight')
plt.show()
print('Ramachandran plot saved to /tmp/ramachandran_plot.png')

### 2.4 Reading the Ramachandran plot

**What you should see:**
- Red cluster near (−57, −47): α-helix residues. Dense, tight cluster = well-formed helix.
- Blue cluster near (−119, +113): β-strand residues. Upper-left quadrant (II in standard notation).
- Grey scatter spread across the plot: coil/loop residues. No preferred conformation.
- Dashed rectangles approximate the 'core allowed' regions from Lovell et al. 2003.

**In a quality structure:**  
>98% of non-Gly, non-Pro residues should fall inside the favoured regions.  
MolProbity flags structures where >0.5% of residues are in 'disallowed' space.

**Interview answer:** "A Ramachandran plot shows the distribution of backbone phi/psi dihedral angles. Allowed regions correspond to geometries that avoid steric clashes between backbone atoms. Outliers indicate modelling errors or genuine but unusual conformations."


---
## 3. DSSP: Simplified Secondary Structure Assignment

### 3.1 What is DSSP?

DSSP (Define Secondary Structure of Proteins, Kabsch & Sander 1983) is the canonical algorithm for assigning secondary structure from 3D coordinates. It uses hydrogen bond pattern detection. Here we implement a **simplified rule-based version** based solely on phi/psi angles — sufficient for understanding the concept and for building ML features.

### 3.2 Assignment rules (simplified)

| Assignment | Phi (°) | Psi (°) | Mnemonic |
|-----------|---------|---------|----------|
| H (helix) | −90 to −30 | −75 to −5 | Both angles negative, modest magnitude |
| E (strand) | −180 to −40 | +90 to +180 | phi very negative, psi large positive |
| C (coil) | everything else | | Default |

Full DSSP additionally uses: hydrogen bond energy, hydrogen bond patterns (3-10 helix, π-helix), and geometrical criteria for bridges. For ML feature extraction the simplified version captures the main signal.


In [ ]:
# ── Simplified DSSP assignment ────────────────────────────────────────────────

def assign_ss_simple(phi, psi):
    """
    Assign secondary structure code from phi/psi angles (degrees).

    Rules
    -----
    H : phi in [-90, -30] AND psi in [-75, -5]     (alpha-helix)
    E : phi in [-180, -40] AND psi in [+90, +180]  (beta-strand)
    C : all other                                   (coil/loop)

    Parameters
    ----------
    phi : float   dihedral angle in degrees
    psi : float   dihedral angle in degrees

    Returns
    -------
    str : 'H', 'E', or 'C'
    """
    if (-90 <= phi <= -30) and (-75 <= psi <= -5):
        return 'H'
    elif (-180 <= phi <= -40) and (90 <= psi <= 180):
        return 'E'
    else:
        return 'C'


def assign_ss_array(phi_arr, psi_arr):
    """Vectorised wrapper: returns list of 'H'/'E'/'C' codes."""
    return [assign_ss_simple(p, q) for p, q in zip(phi_arr, psi_arr)]


def ss_composition(ss_codes):
    """Return fractional composition {'H': fH, 'E': fE, 'C': fC}."""
    n = len(ss_codes)
    counts = {k: ss_codes.count(k) / n for k in ['H', 'E', 'C']}
    return counts


# Assign SS for each simulated protein
ss_helix  = assign_ss_array(phi_h.tolist(), psi_h.tolist())
ss_strand = assign_ss_array(phi_s.tolist(), psi_s.tolist())
ss_coil   = assign_ss_array(phi_c.tolist(), psi_c.tolist())

print('=== Secondary Structure Composition ===')
print(f'\nHelix-rich protein   (ground truth: helix):')
comp_h = ss_composition(ss_helix)
for code, frac in comp_h.items():
    print(f'  {code}: {frac*100:.1f}%')

print(f'\nStrand-rich protein  (ground truth: strand):')
comp_s = ss_composition(ss_strand)
for code, frac in comp_s.items():
    print(f'  {code}: {frac*100:.1f}%')

print(f'\nCoil-rich protein    (ground truth: coil):')
comp_c = ss_composition(ss_coil)
for code, frac in comp_c.items():
    print(f'  {code}: {frac*100:.1f}%')

In [ ]:
# ── Visualise SS composition as a stacked bar chart ───────────────────────────

proteins   = ['Helix-rich', 'Strand-rich', 'Coil-rich']
comp_list  = [comp_h, comp_s, comp_c]
ss_labels  = ['H (helix)', 'E (strand)', 'C (coil)']
colors     = ['crimson', 'royalblue', 'lightgrey']
keys       = ['H', 'E', 'C']

fig, ax = plt.subplots(figsize=(8, 5))

bar_width = 0.5
bottoms   = np.zeros(len(proteins))

for key, label, color in zip(keys, ss_labels, colors):
    values = [c[key] for c in comp_list]
    ax.bar(proteins, values, bar_width, bottom=bottoms,
           label=label, color=color, edgecolor='white')
    bottoms += np.array(values)

ax.set_ylim(0, 1.05)
ax.set_ylabel('Fraction of residues', fontsize=12)
ax.set_title('Simplified DSSP: Secondary Structure Composition per Protein', fontsize=11)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print('\nInterpretation:')
print('  The helix-rich protein shows ~70-90% H.')
print('  The strand-rich protein shows ~50-70% E.')
print('  The coil protein shows mostly C with scattered H and E.')
print('  Residuals arise because the simplified rule has narrower bins than real DSSP.')

### 3.3 From DSSP to ML features

Secondary structure assignment generates a **per-residue categorical label** that is used as:
1. A **node feature** in protein graph neural networks (Module 06): each node carries a one-hot vector [H, E, C]
2. A **sequence annotation** for sequence models: the SS string is a secondary sequence input
3. A **design target** in protein design models: generate sequences that fold into a desired SS pattern

**Composition vector** (3D: %H, %E, %C) is a simple but powerful protein fingerprint — used in early ML classifiers for protein fold prediction.


---
## 4. Contact Maps: Cα Distance Matrix and ML Features

### 4.1 Definition

A **contact map** is an N×N binary matrix where entry (i,j) = 1 if residues i and j are in 'contact' (Cα atoms within a threshold distance, typically **8 Å**) and 0 otherwise.

The underlying **distance matrix** D where D[i,j] = ||Cα_i − Cα_j||₂ encodes full pairwise distance information.

### 4.2 Why the 8 Å threshold?

- At 8 Å, Cα atoms are close enough that their side chains can interact (side chain lengths are typically 1–5 Å).
- Residues within 3–4 in sequence are always in contact (sequential neighbours along the chain) — these are usually **masked** in ML applications to focus on long-range contacts.
- AlphaFold 2 predicted a full **distogram** (probability distribution over distance bins) rather than a binary contact map — this is strictly more informative.

### 4.3 Contact maps as GNN input (connection to Module 06)

In Module 06 you will build protein graph neural networks where:
- **Nodes** = residues (with features: amino acid identity, secondary structure, B-factor, ...)
- **Edges** = contacts (contact map threshold defines which pairs are connected)
- **Edge features** = distance, relative orientation of Cβ vectors

The contact map is literally the **adjacency matrix** of the protein graph.


In [ ]:
# ── Simulate Cα coordinates for a mixed secondary structure protein ───────────

def simulate_ca_coords(n_residues=100, seed=0):
    """
    Simulate Cα coordinates for a protein with:
      - residues  0-29 : alpha-helix  (tight coil in 3D)
      - residues 30-59 : beta-sheet   (extended, planar)
      - residues 60-99 : loop/coil    (random walk)

    Returns
    -------
    ca : np.ndarray shape (n_residues, 3)
    ss_labels : list of 'H'/'E'/'C' per residue
    """
    rng = np.random.default_rng(seed)
    ca  = np.zeros((n_residues, 3))
    ss_labels = []

    # Segment 1: alpha-helix — rise 1.5 Å/residue, radius 2.3 Å, pitch
    for i in range(30):
        angle = i * np.radians(100)   # 100 degrees per residue in ideal helix
        ca[i] = [2.3 * np.cos(angle),
                 2.3 * np.sin(angle),
                 i * 1.5]
        ss_labels.append('H')

    # Segment 2: beta-strand — extended, ~3.5 Å between Cα along x-axis
    start = ca[29] + np.array([3.5, 0, 0])
    for i in range(30):
        ca[30 + i] = start + np.array([i * 3.5,
                                       0.5 * ((-1)**i),    # slight zigzag
                                       0])
        ss_labels.append('E')

    # Segment 3: random coil — random walk with step ~3.8 Å
    pos = ca[59].copy()
    for i in range(40):
        step = rng.standard_normal(3)
        step = step / np.linalg.norm(step) * 3.8
        pos  = pos + step
        ca[60 + i] = pos
        ss_labels.append('C')

    return ca, ss_labels


ca_coords, ss_labels_mixed = simulate_ca_coords(n_residues=100)
print(f'Cα coordinates shape: {ca_coords.shape}')
print(f'SS label count: H={ss_labels_mixed.count("H")}, '
      f'E={ss_labels_mixed.count("E")}, '
      f'C={ss_labels_mixed.count("C")}')

In [ ]:
# ── Compute distance matrix and contact map ───────────────────────────────────

def compute_distance_matrix(ca):
    """
    Compute pairwise Cα distance matrix.

    Parameters
    ----------
    ca : np.ndarray shape (N, 3)

    Returns
    -------
    D : np.ndarray shape (N, N), dtype float
        D[i,j] = Euclidean distance in Angstroms between Ca_i and Ca_j

    Implementation note
    -------------------
    Using broadcasting: ||a - b||^2 = ||a||^2 + ||b||^2 - 2*a.b
    This avoids an explicit O(N^2) loop.
    """
    # Squared norms for each Cα
    sq_norms = np.sum(ca ** 2, axis=1, keepdims=True)   # shape (N,1)
    # Squared distance matrix via broadcasting
    sq_dist = sq_norms + sq_norms.T - 2.0 * (ca @ ca.T)
    # Numerical safety: clip negative values (floating-point noise on diagonal)
    sq_dist = np.clip(sq_dist, 0, None)
    return np.sqrt(sq_dist)


def compute_contact_map(dist_matrix, threshold_angstrom=8.0, min_seq_sep=4):
    """
    Binary contact map from distance matrix.

    Parameters
    ----------
    dist_matrix     : (N, N) float array
    threshold_angstrom : float  distance cutoff (default 8.0 Å)
    min_seq_sep     : int   mask contacts |i-j| < min_seq_sep (sequential neighbours)

    Returns
    -------
    contact_map : (N, N) bool array
    """
    N = dist_matrix.shape[0]
    contact_map = (dist_matrix < threshold_angstrom).astype(int)

    # Mask diagonal and near-diagonal (sequential neighbours are trivially in contact)
    for k in range(-min_seq_sep + 1, min_seq_sep):
        idx = np.arange(max(0, -k), min(N, N - k))
        contact_map[idx, idx + k] = 0

    return contact_map


D_matrix = compute_distance_matrix(ca_coords)
C_map    = compute_contact_map(D_matrix, threshold_angstrom=8.0, min_seq_sep=4)

print(f'Distance matrix shape : {D_matrix.shape}')
print(f'Min distance (off-diag): {D_matrix[D_matrix > 0].min():.2f} Angstrom')
print(f'Max distance           : {D_matrix.max():.2f} Angstrom')
print(f'Contact map sparsity   : {C_map.sum()} contacts out of {100*99} possible pairs')
print(f'Contact density        : {C_map.sum() / (100*99) * 100:.1f}%')

In [ ]:
# ── Visualise distance matrix and contact map side by side ────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: Distance matrix
ax1 = axes[0]
im1 = ax1.imshow(D_matrix, cmap='viridis_r', aspect='auto')
plt.colorbar(im1, ax=ax1, label='Cα-Cα distance (Å)')
ax1.set_title('Cα Distance Matrix', fontsize=12)
ax1.set_xlabel('Residue index')
ax1.set_ylabel('Residue index')

# Add secondary structure annotation on axes
ss_colors_map = {'H': 'crimson', 'E': 'royalblue', 'C': 'lightgrey'}
for i, ss in enumerate(ss_labels_mixed):
    ax1.axvline(x=i, color=ss_colors_map[ss], alpha=0.04, linewidth=1)

# Panel 2: Contact map (binary)
ax2 = axes[1]
im2 = ax2.imshow(C_map, cmap='Greys', aspect='auto', vmin=0, vmax=1)
ax2.set_title('Contact Map (8 Å threshold, seq sep ≥ 4)', fontsize=12)
ax2.set_xlabel('Residue index')
ax2.set_ylabel('Residue index')

# Mark secondary structure regions
for ax in [ax2]:
    ax.axhline(y=29.5, color='crimson', linestyle='--', linewidth=1, alpha=0.7, label='helix end')
    ax.axhline(y=59.5, color='royalblue', linestyle='--', linewidth=1, alpha=0.7, label='strand end')
    ax.axvline(x=29.5, color='crimson', linestyle='--', linewidth=1, alpha=0.7)
    ax.axvline(x=59.5, color='royalblue', linestyle='--', linewidth=1, alpha=0.7)
    ax.legend(fontsize=8)

# Panel 3: Annotated contact map with SS colour strip
ax3 = axes[2]
im3 = ax3.imshow(C_map, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax3.set_title('Contact Map — structural annotations', fontsize=12)
ax3.set_xlabel('Residue index')
ax3.set_ylabel('Residue index')

# Colour strip on top
ss_strip = np.array([[{'H': [220,20,60],
                        'E': [65,105,225],
                        'C': [200,200,200]}[s] for s in ss_labels_mixed]])
ax3_inset = ax3.inset_axes([0, 1.01, 1, 0.03], transform=ax3.transAxes)
ax3_inset.imshow(ss_strip, aspect='auto')
ax3_inset.set_axis_off()

# Add text annotations for structural patterns
ax3.text(15, 15, 'Helix\nlocal\ncontacts', ha='center', fontsize=7,
         color='white', fontweight='bold')
ax3.text(45, 45, 'Strand\nlocal\ncontacts', ha='center', fontsize=7,
         color='white', fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/contact_map.png', dpi=120, bbox_inches='tight')
plt.show()
print('Contact map saved to /tmp/contact_map.png')

### 4.4 Structural patterns in contact maps

| Pattern | Appearance | Secondary structure |
|---------|-----------|---------------------|
| Dense near-diagonal band | Stripe parallel to diagonal | α-helix (contacts at i±3, i±4) |
| Off-diagonal stripe parallel to diagonal | Long stripe | Parallel β-sheet |
| Off-diagonal stripe perpendicular to diagonal | Anti-parallel stripe | Anti-parallel β-sheet |
| Sparse, sparse blocks | Random scatter | Loop/coil regions |
| Block of contacts between residue ranges | Dense rectangle | Domain-domain interface |

**GNN connection (Module 06):**  
When you construct a protein graph, the contact map is the adjacency matrix. Message passing aggregates information from *contacting* residues — exactly the residues that could physically interact. This is why GNNs outperform sequential RNNs on structure-aware tasks: the edges encode 3D proximity rather than sequence distance.


---
## 5. Distance Matrices as ML Features: SVM Classifier

### 5.1 Feature engineering from the distance matrix

For classification tasks (e.g., predict fold class, predict binding propensity) we need a fixed-length feature vector. The upper triangle of the distance matrix is a natural choice:

- For an N-residue protein: N(N-1)/2 features
- Often we normalise by the maximum pairwise distance
- For variable-length proteins we use padding or a fixed-length summary (contact density per secondary structure, eigenvalues of the contact matrix, etc.)

Here we generate a synthetic dataset of helix-rich and strand-rich proteins and classify them.


In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix

# ── Generate synthetic dataset ────────────────────────────────────────────────
# Two classes:
#   Class 0: helix-rich proteins (30 residues, helix geometry)
#   Class 1: strand-rich proteins (30 residues, strand geometry)
# Feature: upper-triangle of Cα distance matrix → 30*29/2 = 435 features

N_PROTEINS  = 60    # 30 per class
N_RES       = 30    # residues per protein (fixed length for simplicity)
N_FEATURES  = N_RES * (N_RES - 1) // 2


def make_helix_ca(n_res, seed):
    """Simulate Cα coordinates for an ideal alpha-helix."""
    rng = np.random.default_rng(seed)
    ca  = np.zeros((n_res, 3))
    for i in range(n_res):
        angle  = i * np.radians(100) + rng.uniform(-0.1, 0.1)
        radius = 2.3 + rng.normal(0, 0.05)
        rise   = 1.5 + rng.normal(0, 0.03)
        ca[i]  = [radius * np.cos(angle),
                  radius * np.sin(angle),
                  i * rise]
    return ca


def make_strand_ca(n_res, seed):
    """Simulate Cα coordinates for an extended beta-strand."""
    rng  = np.random.default_rng(seed)
    ca   = np.zeros((n_res, 3))
    step = 3.5  # Å per residue in extended strand
    for i in range(n_res):
        ca[i] = [i * step + rng.normal(0, 0.1),
                 0.5 * ((-1)**i) + rng.normal(0, 0.05),
                 rng.normal(0, 0.05)]
    return ca


def upper_triangle_features(ca):
    """Compute distance matrix and flatten upper triangle (excluding diagonal)."""
    D   = compute_distance_matrix(ca)
    idx = np.triu_indices_from(D, k=1)  # k=1 excludes diagonal
    feat = D[idx]
    # Normalise by maximum distance
    feat = feat / (feat.max() + 1e-8)
    return feat


X_list, y_list = [], []

for i in range(30):
    ca_h = make_helix_ca(N_RES, seed=i)
    X_list.append(upper_triangle_features(ca_h))
    y_list.append(0)   # class 0: helix-rich

for i in range(30):
    ca_s = make_strand_ca(N_RES, seed=i + 100)
    X_list.append(upper_triangle_features(ca_s))
    y_list.append(1)   # class 1: strand-rich

X = np.array(X_list)   # shape (60, 435)
y = np.array(y_list)

print(f'Feature matrix shape: {X.shape}')
print(f'Label distribution:   {dict(zip(*np.unique(y, return_counts=True)))}')

In [ ]:
# ── Train and evaluate SVM ────────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm.fit(X_train_sc, y_train)

y_pred = svm.predict(X_test_sc)

print('=== SVM Classification Report ===')
print(classification_report(y_test, y_pred,
                             target_names=['helix-rich', 'strand-rich']))

cv_scores = cross_val_score(svm, scaler.fit_transform(X), y, cv=5)
print(f'5-fold CV accuracy: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')

print('\nConfusion matrix:')
print(confusion_matrix(y_test, y_pred))
print('  [TN  FP]')
print('  [FN  TP]')

print('\nKey takeaway: distance matrix upper triangle is a simple but effective')
print('structural fingerprint. For real proteins, PCA/kernel PCA is used to')
print('compress the N(N-1)/2 features before SVM/random forest.')

In [ ]:
# ── Visualise: PCA of distance-matrix features ────────────────────────────────

from sklearn.decomposition import PCA

X_sc_all = StandardScaler().fit_transform(X)
pca      = PCA(n_components=2)
X_pca    = pca.fit_transform(X_sc_all)

fig, ax = plt.subplots(figsize=(7, 5))
colors  = np.where(y == 0, 'crimson', 'royalblue')
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.8, s=60, edgecolors='white')

helix_patch  = mpatches.Patch(color='crimson',    label='helix-rich (class 0)')
strand_patch = mpatches.Patch(color='royalblue',  label='strand-rich (class 1)')
ax.legend(handles=[helix_patch, strand_patch])

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)', fontsize=11)
ax.set_title('PCA of Cα Distance Matrix Features\n(helix vs strand proteins)', fontsize=11)
plt.tight_layout()
plt.show()

print(f'PC1 explains {pca.explained_variance_ratio_[0]*100:.1f}% of variance')
print('Perfect or near-perfect separation expected: helix and strand have'
      ' fundamentally different pairwise distance distributions.')

---
## 6. Cryo-EM Concepts (Conceptual — No Implementation)

This section is conceptual. Cryo-EM implementation requires specialised packages (EMAN2, RELION, CryoDRGN) beyond our NumPy scope. Understanding the concepts is required for interviews.

### 6.1 What is cryo-EM?

Cryo-electron microscopy (cryo-EM) determines protein structure by:
1. **Freezing** the sample in vitreous ice (rapid plunge-freezing prevents crystalline ice formation)
2. **Imaging** with an electron beam — each image is a 2D projection of a 3D molecule
3. **Reconstruction** — particles in random orientations are computationally combined into a 3D density map
4. **Model building** — an atomic model is fitted into the density map

### 6.2 Resolution

Resolution in cryo-EM is measured in Ångströms. The **Fourier Shell Correlation (FSC) at 0.143** criterion is the standard measure.

| Resolution | What you can see |
|------------|------------------|
| > 5 Å | Overall shape and domain organisation |
| 3–5 Å | Secondary structure elements (helices, sheets) |
| 2–3 Å | Side chain positions, can build atomic model |
| < 2 Å | Water molecules, detailed electrostatics |

The 'resolution revolution' (2013–2016) pushed cryo-EM from 4–6 Å to 2–3 Å routinely, driven by improved electron detectors.

### 6.3 Map quality metrics

- **FSC (Fourier Shell Correlation):** Measures consistency between two half-maps. FSC = 0.143 threshold defines the nominal resolution.
- **Map-model correlation coefficient (CC):** How well the atomic model fits into the density. CC > 0.8 is good for 3 Å maps.
- **Local resolution:** Resolution varies across the map — flexible loops have lower local resolution than the core.
- **Q-score:** Per-atom map quality score (Pintilie et al. 2020) — analogous to B-factor but for cryo-EM.

### 6.4 AlphaFold 3 vs cryo-EM density

This is a common interview question:

> **Q: How do AlphaFold 3 predictions compare to cryo-EM experimental structures?**
> 
> **A:** AlphaFold 3 predicts Cα positions with median RMSD ~1 Å vs 3 Å cryo-EM structures for ordered regions. However:  
> (1) AF3 lacks experimental density — it cannot capture alternative conformations or ligand-induced states.  
> (2) Flexible loops and termini have high uncertainty (low pLDDT) in AF3, mirroring poor local resolution in cryo-EM.  
> (3) AF3's pLDDT is strongly correlated with cryo-EM local resolution and B-factors.  
> (4) AF3 predictions are routinely used as **starting models** for cryo-EM map fitting (molecular replacement analogue), dramatically accelerating model building.

### 6.5 Model building pipeline

```
Raw micrographs
    |
    v
Particle picking (RELION / crYOLO)
    |
    v
2D classification --> good 2D classes
    |
    v
3D reconstruction  --> density map (.mrc file)
    |
    v
Model building (COOT, Phenix, ModelAngelo)
    |  <--- AF3 prediction used as starting model here
    v
Real-space refinement --> final atomic model (.pdb file)
    |
    v
Validation (MolProbity, PHENIX validation)
```


---
## 7. Structure Quality Metrics

### 7.1 B-factors (temperature factors)

The B-factor (Debye-Waller factor) in crystallographic and cryo-EM models captures the **displacement uncertainty** of each atom:

$$B = 8\pi^2 \langle u^2 \rangle$$

where $\langle u^2 \rangle$ is the mean-square displacement of the atom from its mean position.

| B-factor range | Interpretation | ML implication |
|---------------|----------------|----------------|
| 10–30 Å² | Well-ordered residue | High confidence, use as-is |
| 30–60 Å² | Moderate disorder | Some uncertainty, consider down-weighting |
| > 60 Å² | Highly disordered | Low confidence, often masked in training |
| > 100 Å² | Likely mis-modelled | Exclude from training |

**Connection to AlphaFold 3:**  
pLDDT (predicted Local Distance Difference Test, 0–100 scale) is AF3's per-residue confidence score. It is strongly *anti-correlated* with B-factor: high pLDDT → ordered region → low B-factor.

### 7.2 MolProbity score

MolProbity (Davis et al. 2007, doi:10.1093/nar/gkp431) integrates:
- **Clashscore**: number of serious steric clashes per 1000 atoms (target: < 10)
- **Ramachandran outliers**: % residues in disallowed regions (target: < 0.5%)
- **Rotamer outliers**: % side chain conformations in disallowed rotameric states (target: < 1%)

The combined MolProbity score is a single number where lower is better. Structures deposited in the PDB typically have MolProbity < 2.0.

### 7.3 Clash detection

A **steric clash** occurs when two non-bonded atoms are closer than the sum of their Van der Waals radii minus a tolerance. For a simplified check: any two non-bonded heavy atoms with distance < 1.5 Å are clashing.


In [ ]:
# ── Simulate B-factors and analyse ────────────────────────────────────────────

def simulate_bfactors(n_residues=100, seed=7):
    """
    Simulate realistic B-factor profile:
    - Core: B ~ 20-30 (ordered)
    - N/C termini: B ~ 50-80 (disordered)
    - Loop regions: B ~ 40-70
    """
    rng = np.random.default_rng(seed)
    # Base B-factor: U-shaped (higher at termini)
    x = np.linspace(0, 1, n_residues)
    b_base = 20 + 60 * (x - 0.5)**2 + rng.normal(0, 5, n_residues)

    # Add two loop regions (residues 40-50 and 75-85)
    b_base[40:51] += rng.uniform(20, 40, 11)
    b_base[75:86] += rng.uniform(15, 35, 11)

    return np.clip(b_base, 5, 120)


bfactors = simulate_bfactors(n_residues=100)

# Classify residues by B-factor
ordered    = np.sum(bfactors < 30)
moderate   = np.sum((bfactors >= 30) & (bfactors < 60))
disordered = np.sum(bfactors >= 60)

print(f'B-factor statistics:')
print(f'  Mean: {bfactors.mean():.1f}  Std: {bfactors.std():.1f}  Min: {bfactors.min():.1f}  Max: {bfactors.max():.1f}')
print(f'  Ordered    (B < 30):  {ordered:3d} residues ({ordered}%)')
print(f'  Moderate   (30-60):   {moderate:3d} residues ({moderate}%)')
print(f'  Disordered (B >= 60): {disordered:3d} residues ({disordered}%)')

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
residues = np.arange(100)

# Colour bars by B-factor category
bar_colors = ['crimson' if b >= 60 else 'orange' if b >= 30 else 'steelblue'
              for b in bfactors]
ax.bar(residues, bfactors, color=bar_colors, width=1.0, alpha=0.8)

ax.axhline(30, color='orange', linestyle='--', linewidth=1.5, label='B=30 threshold')
ax.axhline(60, color='crimson', linestyle='--', linewidth=1.5, label='B=60 threshold')

# Annotate loop regions
ax.axvspan(40, 50, alpha=0.15, color='crimson', label='Loop region 1')
ax.axvspan(75, 85, alpha=0.15, color='crimson', label='Loop region 2')

ax.set_xlabel('Residue index', fontsize=11)
ax.set_ylabel('B-factor (Å²)', fontsize=11)
ax.set_title('Per-residue B-factor Profile\n(blue=ordered, orange=moderate, red=disordered)',
             fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Clash detection: atoms closer than 1.5 Å ─────────────────────────────────

def detect_clashes(atom_coords, clash_threshold=1.5, bond_exclusion_mask=None):
    """
    Find all pairs of atoms with distance < clash_threshold.

    Parameters
    ----------
    atom_coords       : np.ndarray shape (M, 3)  — M atoms in 3D space
    clash_threshold   : float  in Angstroms (default 1.5 Å)
    bond_exclusion_mask : np.ndarray shape (M, M) bool — if True, skip that pair
                          (use to exclude bonded 1-2 and 1-3 pairs)

    Returns
    -------
    clashes : list of tuples (i, j, distance)
    clashscore : float  clashes per 1000 atoms
    """
    M = atom_coords.shape[0]

    # Pairwise distance matrix
    sq_norms = np.sum(atom_coords ** 2, axis=1, keepdims=True)
    sq_dist  = sq_norms + sq_norms.T - 2.0 * (atom_coords @ atom_coords.T)
    sq_dist  = np.clip(sq_dist, 0, None)
    dist_mat = np.sqrt(sq_dist)

    # Mask diagonal (self-distances = 0) and lower triangle
    upper_mask = np.triu(np.ones((M, M), dtype=bool), k=1)

    # Apply bond exclusion if provided
    if bond_exclusion_mask is not None:
        upper_mask = upper_mask & (~bond_exclusion_mask)

    # Find clashes
    clash_mask = (dist_mat < clash_threshold) & upper_mask
    i_idx, j_idx = np.where(clash_mask)

    clashes = [(int(i), int(j), float(dist_mat[i, j]))
               for i, j in zip(i_idx, j_idx)]

    clashscore = len(clashes) / M * 1000.0
    return clashes, clashscore


# ── Simulate atom positions with intentional clashes ─────────────────────────
# Create a small mock structure: 50 atoms with mostly proper spacing
# but 3 deliberately introduced clashes

rng = np.random.default_rng(99)
N_ATOMS = 50

# Place atoms on a rough grid (2 Å spacing) with small noise
grid_pts = np.array([[x, y, 0.0]
                     for x in np.arange(0, 20, 2.0)
                     for y in np.arange(0, 6, 2.0)])[:N_ATOMS].copy()
grid_pts += rng.normal(0, 0.1, grid_pts.shape)

# Introduce 3 deliberate clashes by moving atoms very close to existing ones
clash_indices = [5, 15, 35]
for ci in clash_indices:
    target = rng.choice([j for j in range(N_ATOMS) if j != ci])
    grid_pts[ci] = grid_pts[target] + rng.normal(0, 0.3, 3)

clashes, cs = detect_clashes(grid_pts, clash_threshold=1.5)

print(f'Total atoms: {N_ATOMS}')
print(f'Clashes detected (distance < 1.5 Å): {len(clashes)}')
print(f'Clashscore: {cs:.1f} clashes per 1000 atoms')
print()
for i, j, d in clashes[:10]:
    print(f'  Atoms {i:3d} and {j:3d}: {d:.3f} Å')

print()
print('MolProbity clashscore interpretation:')
print('  < 10  : good')
print('  10-40 : acceptable')
print('  > 40  : poor, likely modelling errors')
print(f'  This mock structure: {cs:.1f} — {"good" if cs < 10 else "acceptable" if cs < 40 else "poor"}')

In [ ]:
# ── Ramachandran outlier percentage calculation ───────────────────────────────

def ramachandran_outlier_pct(phi_arr, psi_arr,
                              allowed_regions=None):
    """
    Compute the percentage of phi/psi pairs that fall outside allowed regions.

    Simplified allowed regions (rectangles):
      alpha-helix:    phi in [-90, -30], psi in [-75,  -5]
      beta-strand:    phi in [-180,-40], psi in [ 90, 180]
      left-hand helix: phi in [30, 90],  psi in [ 20,  80]  (Gly only, approximate)
      polyproline II: phi in [-90, -50], psi in [120, 170]  (extended helix)

    Returns
    -------
    pct_outlier : float  (0–100)
    n_outliers  : int
    """
    if allowed_regions is None:
        allowed_regions = [
            # (phi_lo, phi_hi, psi_lo, psi_hi)
            (-90,  -30, -75,  -5),    # alpha-helix
            (-180, -40,  90, 180),    # beta-strand (right half)
            ( 30,   90,  20,  80),    # left-hand helix
            (-90,  -50, 120, 170),    # PPII
            (-180, -40, -180, -90),   # lower left quadrant (also allowed)
        ]

    n_total    = len(phi_arr)
    n_outliers = 0

    for phi, psi in zip(phi_arr, psi_arr):
        in_allowed = False
        for (pl, ph, ql, qh) in allowed_regions:
            if pl <= phi <= ph and ql <= psi <= qh:
                in_allowed = True
                break
        if not in_allowed:
            n_outliers += 1

    return (n_outliers / n_total * 100), n_outliers


pct_h, n_h = ramachandran_outlier_pct(phi_h, psi_h)
pct_s, n_s = ramachandran_outlier_pct(phi_s, psi_s)
pct_c, n_c = ramachandran_outlier_pct(phi_c, psi_c)

print('=== Ramachandran Outlier Analysis ===')
print(f'Helix protein:  {pct_h:.1f}% outliers ({n_h}/{len(phi_h)} residues)')
print(f'Strand protein: {pct_s:.1f}% outliers ({n_s}/{len(phi_s)} residues)')
print(f'Coil protein:   {pct_c:.1f}% outliers ({n_c}/{len(phi_c)} residues)')
print()
print('Expected: helix and strand proteins should have low outlier rates')
print('because their phi/psi angles are within well-defined allowed regions.')
print('Coil proteins have random angles -> high outlier rate in simplified model.')
print()
print('MolProbity threshold: >0.5% outliers flags a structure as needing refinement.')

---
## 8. Interview Questions and Answers

Five questions you are likely to encounter in ML engineer interviews at Isomorphic Labs, DeepMind, or Schrödinger.


### Q1. What does a Ramachandran plot tell you, and what does a high outlier rate imply?

**Model answer:**

A Ramachandran plot displays the distribution of backbone phi (N–Cα bond rotation) and psi (Cα–C bond rotation) dihedral angles for every non-glycine, non-proline residue in a protein structure. The plot has well-defined 'allowed' and 'favoured' regions, determined by steric clash analysis (Ramachandran et al., 1963). Alpha-helical residues cluster near (−57°, −47°) and beta-strand residues near (−119°, +113°).

A high outlier rate (>2%) implies:  
(1) modelling errors during structure determination — atoms placed in incorrect positions;  
(2) need for further refinement;  
(3) for ML training data, such a structure should be excluded or its disordered regions masked.

For ML specifically: training a model on Ramachandran-outlier-heavy structures will cause it to learn physically impossible geometries.

---

### Q2. How are contact maps used in graph neural networks for proteins?

**Model answer:**

In a protein GNN (e.g., as used in Module 06), each residue is a **node** with feature vector encoding amino acid identity, secondary structure, B-factor, and surface accessibility. The **contact map defines edges**: two residues are connected by an edge if their Cα atoms are within the threshold distance (typically 8 Å), meaning their side chains can physically interact.

Message passing then aggregates information from neighbouring residues, enabling the model to learn representations that account for 3D proximity rather than just sequence distance. This is why GNNs outperform purely sequence-based models for tasks like binding site prediction, mutation effect prediction (ΔΔG), and secondary structure prediction.

Edge features encode the actual distance, relative backbone orientation, and in more sophisticated models (e.g., IPA in AF3) the full rotation frame. AlphaFold 2's Evoformer operated on a distance-like pair representation that is conceptually the continuous analogue of a contact map.

---

### Q3. What is a B-factor, and how does it relate to AlphaFold confidence (pLDDT)?

**Model answer:**

The B-factor (also called the temperature factor or Debye-Waller factor) parameterises the displacement of an atom from its mean position: B = 8π²⟨u²⟩, where ⟨u²⟩ is the mean-square displacement. It incorporates both genuine thermal motion and static disorder across the crystal unit cells.

High B-factors (>60 Å²) indicate the atom is poorly ordered — either genuinely mobile (loops, termini) or not reliably modelled. In ML, high-B-factor residues are often excluded from training or down-weighted in the loss function.

AlphaFold's pLDDT (0–100) is the predicted confidence in local structure. It is strongly anti-correlated with B-factor: pLDDT > 90 corresponds to ordered regions with B < 30 Å², while pLDDT < 50 corresponds to disordered regions with B > 80 Å². This makes pLDDT directly useful as a quality filter when using AF3 predictions as training data.

---

### Q4. Explain the MolProbity score and its components.

**Model answer:**

MolProbity (Davis et al. 2007) is the standard validation server for protein structures. Its score integrates:

1. **Clashscore:** steric clashes per 1000 atoms (overlap > 0.4 Å). Target < 10.
2. **Ramachandran outliers:** % residues in disallowed regions. Target < 0.5%.
3. **Rotamer outliers:** % side chains in disallowed rotameric conformations. Target < 1%.

The composite MolProbity score is the log-weighted combination, calibrated so that structures deposited in the PDB have MolProbity < 2.0. Lower is better.

For ML: MolProbity score is used as a hard filter when curating training datasets. The PDB filter for AlphaFold training (Jumper et al. 2021) used a combination of resolution < 3.5 Å and MolProbity score thresholding.

---

### Q5. Why is an 8 Å threshold used for contact maps? What happens if you use 6 Å or 12 Å?

**Model answer:**

8 Å is chosen because at this Cα-Cα distance, the side chains of the two residues can interact physically (most side chains are 2–5 Å long). It is the standard in the field (Dunn et al. 2008) and captures both direct contacts and second-shell interactions.

**6 Å threshold:** Only captures very close contacts. The map is sparser, potentially missing long-range interactions important for fold determination. However, it reduces false positives (apparent contacts that are not functionally relevant).

**12 Å threshold:** Very dense map — almost every residue within a typical helix or sheet is connected to every other. The GNN receives too-rich connectivity, making it harder to learn the specificity of interactions. Also: more edges → more memory and compute in the GNN.

**In practice:** Some models use a **distance threshold + top-k contacts** to control graph density. AF3's structure module uses a distance cutoff of ~10 Å combined with kNN graph construction.


---
## 9. Debug Exercise: Three Bugs in Dihedral Angle Calculation

The function below contains **three deliberate bugs**. Find and fix them before running the test cell.

**Hint categories:**
- Bug 1: wrong atom order in bond vector construction
- Bug 2: missing `atan2` (using `arccos` instead loses the sign)
- Bug 3: wrong sign convention on the `y` component


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BUGGY IMPLEMENTATION — three bugs to find and fix
# ─────────────────────────────────────────────────────────────────────────────

def dihedral_angle_buggy(a, b, c, d):
    """
    BUGGY version — DO NOT USE in production.
    Contains 3 deliberate errors for the debug exercise.
    """
    # BUG 1: bond vectors defined in wrong direction
    # Correct: b1 = b - a, b2 = c - b, b3 = d - c
    b1 = a - b        # <-- BUG: should be b - a
    b2 = c - b
    b3 = d - c

    n1 = np.cross(b1, b2)
    n2 = np.cross(b2, b3)

    b2_norm = b2 / (np.linalg.norm(b2) + 1e-12)

    x_comp = np.dot(n1, n2)

    # BUG 2: using arccos instead of atan2 — loses sign information
    # Correct: need both x and y components → atan2(y, x)
    norm_n1 = np.linalg.norm(n1) + 1e-12
    norm_n2 = np.linalg.norm(n2) + 1e-12
    angle_rad = np.arccos(np.clip(x_comp / (norm_n1 * norm_n2), -1, 1))  # <-- BUG

    # BUG 3: wrong sign on y component of atan2 (would negate the angle)
    # Correct: y = dot(cross(b2_norm, n1), n2)  — NOT negated
    # (This bug is masked here because we didn't compute y at all — fix Bug 2 first)

    return np.degrees(angle_rad)


# ─────────────────────────────────────────────────────────────────────────────
# FIXED IMPLEMENTATION — fill in the corrections
# ─────────────────────────────────────────────────────────────────────────────

def dihedral_angle_fixed(a, b, c, d):
    """
    Corrected dihedral angle calculation.

    Fix 1: b1 = b - a  (not a - b)
    Fix 2: use atan2(y, x) to get signed angle in [-180, +180]
    Fix 3: y = dot(cross(b2_norm, n1), n2)  — positive, not negated
    """
    # FIX 1: correct bond vector direction
    b1 = b - a        # was: a - b
    b2 = c - b
    b3 = d - c

    n1 = np.cross(b1, b2)
    n2 = np.cross(b2, b3)

    b2_norm = b2 / (np.linalg.norm(b2) + 1e-12)

    x_comp = np.dot(n1, n2)

    # FIX 2: use atan2 with both x and y components
    # FIX 3: y = dot(cross(b2_norm, n1), n2)  — positive sign
    y_comp = np.dot(np.cross(b2_norm, n1), n2)   # was: missing / negated

    return np.degrees(np.arctan2(y_comp, x_comp))


# ── Test: known dihedral angles ───────────────────────────────────────────────
# Alpha-helix ideal phi = -57 degrees
# Use a standard geometric construction for the test

def test_known_dihedral():
    """
    Test dihedral_angle_fixed against analytically known values.

    For a flat molecule in the xy-plane (z=0):
      A=(0,0,0), B=(1,0,0), C=(2,0,0), D=(3,0,0)
      All in a line -> dihedral is undefined (but atan2(0,0) -> 0)

    Test case: trans configuration (180 degrees)
      A=(0, 1,0), B=(0,0,0), C=(1,0,0), D=(1,-1,0)
      → dihedral = 180 degrees (all in xy-plane, trans)
    """
    # Test 1: 0-degree dihedral (eclipsed cis)
    a = np.array([0.0,  1.0, 0.0])
    b = np.array([0.0,  0.0, 0.0])
    c = np.array([1.0,  0.0, 0.0])
    d = np.array([1.0,  1.0, 0.0])
    angle_cis = dihedral_angle_fixed(a, b, c, d)

    # Test 2: 180-degree dihedral (trans)
    d2 = np.array([1.0, -1.0, 0.0])
    angle_trans = dihedral_angle_fixed(a, b, c, d2)

    # Test 3: 90-degree dihedral
    d3 = np.array([1.0, 0.0, 1.0])
    angle_90 = dihedral_angle_fixed(a, b, c, d3)

    print('=== Dihedral Angle Tests ===')
    print(f'Cis  (expected   0°): {angle_cis:.2f}°  {"PASS" if abs(angle_cis) < 1 else "FAIL"}')
    print(f'Trans (expected 180°): {abs(angle_trans):.2f}°  {"PASS" if abs(abs(angle_trans) - 180) < 1 else "FAIL"}')
    print(f'90 deg (expected 90°): {angle_90:.2f}°  {"PASS" if abs(angle_90 - 90) < 1 else "FAIL"}')

    # Test against reference function
    ref = dihedral_angle(a, b, c, d2)
    print(f'\nFixed vs reference (trans): fixed={angle_trans:.2f}°, ref={ref:.2f}°  '
          f'{"PASS" if abs(angle_trans - ref) < 0.1 else "FAIL"}')

    # Show bug: buggy function loses sign
    buggy_90    = dihedral_angle_buggy(a, b, c, d3)
    buggy_trans = dihedral_angle_buggy(a, b, c, d2)
    print(f'\nBuggy function (shows always positive, cannot distinguish +/- angles):')
    print(f'  trans: {buggy_trans:.2f}° (should be ±180°, sign unknown)')
    print(f'  90deg: {buggy_90:.2f}° (should be +90°, bug 1 may give wrong sign)')

test_known_dihedral()

### Bug summary

| Bug | Location | Error | Consequence |
|-----|----------|-------|-------------|
| 1 | Bond vector `b1 = a - b` | Wrong direction: should be `b - a` | Flips plane normal n1, producing wrong sign on all dihedrals |
| 2 | `arccos` instead of `atan2` | `arccos` only returns [0, π], losing the sign | All negative dihedrals appear positive — helix (−57°) looks like +57° |
| 3 | Missing `y` component | Without `y = dot(cross(b2_norm, n1), n2)`, `atan2` cannot be called | Angle always lands in [0, π] quadrant |

**Why the sign matters:** In Ramachandran space, −57° and +57° are in completely different regions. Getting the sign wrong would put helix residues in the 'left-handed helix' region — a critical error for quality assessment.


---
## 10. Resources with DOIs

### Primary references

| Reference | Topic | DOI / Link |
|-----------|-------|------------|
| Ramachandran, Ramakrishnan & Sasisekharan (1963) | Original Ramachandran plot paper | *J Mol Biol* 7:95–99. doi:10.1016/S0022-2836(63)80023-6 |
| Kabsch & Sander (1983) | DSSP algorithm | *Biopolymers* 22:2577–2637. doi:10.1002/bip.360221211 |
| Davis et al. (2007) | MolProbity server | *Nucleic Acids Res* 35:W375. doi:10.1093/nar/gkm216 |
| Chen et al. (2010) | MolProbity score update | *Acta Cryst D* 66:12. doi:10.1107/S0907444909042073 |
| Lovell et al. (2003) | Richardson lab Ramachandran analysis | *Proteins* 50:437. doi:10.1002/prot.10286 |
| Jumper et al. (2021) | AlphaFold 2 | *Nature* 596:583. doi:10.1038/s41586-021-03819-2 |
| Abramson et al. (2024) | AlphaFold 3 | *Nature* 630:493. doi:10.1038/s41586-024-07487-w |
| Pintilie et al. (2020) | Q-score cryo-EM | *Nat Methods* 17:328. doi:10.1038/s41592-020-0731-1 |
| Henderson et al. (1990) | Resolution revolution cryo-EM precursor | *J Mol Biol* 213:899. doi:10.1016/S0022-2836(05)80271-2 |
| Kühlbrandt (2014) | The resolution revolution (cryo-EM review) | *Science* 343:1443. doi:10.1126/science.1251652 |

### Recommended reading path

**Beginner (start here):**
1. Kühlbrandt 2014 (Science, 3 pages) — cryo-EM revolution overview
2. Jumper et al. 2021 (Nature, AF2) — Extended Data Fig 1 for structure quality analysis

**Intermediate:**
3. Lovell et al. 2003 — Ramachandran plot allowed regions (reference for the exact boundaries)
4. Chen et al. 2010 — MolProbity score formula and interpretation

**Advanced:**
5. Abramson et al. 2024 (AF3) — Supplementary Methods for pLDDT definition and training filter criteria

### Tools and databases

- **MolProbity web server:** https://molprobity.biochem.duke.edu/
- **RCSB PDB validation reports:** https://www.rcsb.org/ (every structure has a built-in validation PDF)
- **wwPDB validation pipeline:** https://validate-rcsb-1.wwpdb.org/
- **DSSP online:** https://www.cmbi.umcn.nl/dssp/


---
## 11. Mastery Check

Five questions to test your understanding. Try to answer before looking back.

---

**Question 1.**  
An experimentally-determined protein structure has 3% of residues in the Ramachandran disallowed region. You are considering using it as training data for a GNN that predicts binding affinity. What do you do?

<details>
<summary>Answer</summary>

3% outliers is above the MolProbity threshold of 0.5%. Options:
1. **Exclude the structure** if there are plenty of high-quality alternatives.
2. **Mask outlier residues**: identify the 3% outlier residues, set their node features to a missing-data token, and use a loss mask so the model does not learn from them.
3. **Re-refine the structure** using PHENIX or REFMAC5 before training.
4. **Down-weight** the structure in the training loss by a factor proportional to 1 - outlier_pct.

Never include it as-is without at minimum masking the outlier residues.
</details>

---

**Question 2.**  
You compute the distance matrix for a 200-residue protein. How many elements are in the upper triangle (excluding the diagonal)? If you want to use this as a fixed-length feature vector, what are the concerns?

<details>
<summary>Answer</summary>

Upper triangle elements = N(N-1)/2 = 200 × 199 / 2 = **19,900 features**.

Concerns:
1. **High dimensionality**: 19,900 features for one protein. With 10,000 proteins in training set: 199M entries. Requires PCA or kernel compression.
2. **Variable length**: a 100-residue protein has 4,950 features, incompatible with a 200-residue protein's 19,900 features. Need padding to max length or a length-invariant representation (e.g., eigenvalues of the contact matrix, graph pooling).
3. **Sequential correlations**: nearby residues are always close — the first k(k-1)/2 features are trivially small. Consider masking |i-j| < 5 contacts.
</details>

---

**Question 3.**  
Explain the computational complexity of the distance matrix calculation and describe two approaches to speed it up for very large proteins (e.g., 5000 residues).

<details>
<summary>Answer</summary>

Naive: O(N²) pairs, each requiring O(1) distance computation → **O(N²)** time and space.  
For N=5000: 25M distance calculations, 25M × 4 bytes (float32) = 100 MB matrix — feasible but large.

Speedups:
1. **Vectorised NumPy broadcasting** (what we used): `||a-b||² = ||a||² + ||b||² - 2a·b`. The dot product `A @ A.T` leverages BLAS and is much faster than an explicit loop. Still O(N²) but with a small constant.
2. **Spatial hashing / voxel grid**: partition space into cubes of side = threshold (8 Å). Only check atom pairs within neighbouring cubes. Average O(N) for sparse proteins at a fixed contact density.
3. **Approximate nearest neighbours** (e.g., FAISS): sublinear time for finding k nearest neighbours. Used in large-scale protein structure databases.
4. **Only compute the contact map** (not full distance matrix): with a kD-tree, finding all pairs within 8 Å is O(N log N).
</details>

---

**Question 4.**  
A residue has phi = −130°, psi = +140°. Which secondary structure does the simplified DSSP rule assign it, and is it in the Ramachandran favoured region?

<details>
<summary>Answer</summary>

Simplified DSSP rule: phi ∈ [−180, −40] and psi ∈ [90, 180] → **E (beta-strand)**.

Ramachandran: this point is at (−130°, +140°), within the beta-strand allowed region (upper-left quadrant). It is in the **favoured** region of the Ramachandran plot. A structure with many residues here would have low Ramachandran outlier percentage.
</details>

---

**Question 5.**  
In AlphaFold 3, the pLDDT score for a residue is 35/100. What does this tell you, and how would you use this information when preparing training data for a downstream task?

<details>
<summary>Answer</summary>

pLDDT = 35 is very low confidence (scale: >90 = very high, 70–90 = high, 50–70 = low, <50 = very low). This residue is likely in a disordered or intrinsically unstructured region. Its predicted coordinates are unreliable.

For downstream training data preparation:
1. **Mask the residue**: replace its node features with a learned mask token; exclude it from the loss.
2. **Filter the structure**: if >30% of residues have pLDDT < 50, discard the entire structure.
3. **Use pLDDT as a weight**: loss weight = pLDDT / 100 for each residue, so low-confidence residues contribute less to training.
4. **Flag as disordered**: add an 'is_disordered' binary feature to the node — the model may learn to handle disordered regions differently.

This mirrors how cryo-EM local resolution maps are used: low-resolution regions in the density map get lower weight in model building and validation.
</details>


---
## 12. Cross-References and Connection Map

### Backward dependencies (what you needed to understand this notebook)

| Module | Notebook | Concept used here |
|--------|----------|-------------------|
| 00 | 01_python_ml_basics | NumPy broadcasting, array operations |
| 00 | 04_classification | SVM, StandardScaler, cross_val_score |
| 03 | **01_structure_analysis** | PDB format, Cα extraction, distance matrix, RMSD, Kabsch |

### Forward connections (what this notebook enables)

| Module | Notebook | How this notebook feeds in |
|--------|----------|---------------------------|
| 06 | 01_gnns_for_proteins | **Contact map = adjacency matrix** of protein graph; distance features = edge weights |
| 06 | 02_equivariant_gnns | Secondary structure labels used as initial node features |
| 07 | 01_alphafold3_intro | pLDDT interpretation (B-factor analogue); Ramachandran analysis of AF3 predictions |
| 07 | 04_training_loop | Quality filtering criteria — Ramachandran outliers, clashscore used to curate PDB training set |
| 10 | 01_openfold_walkthrough | MolProbity score used to evaluate fine-tuned model quality |
| 16 | 01_mlops | Structure quality metrics logged as training-data provenance metadata |

### Concept connection diagram

```
03/01: PDB parsing, RMSD, Kabsch
           |           |
           |           +---> Cα coordinates (this notebook)
           |                        |
           |         +--------------+------------------+
           |         |              |                  |
           |    phi/psi        distance matrix    B-factors
           |    Ramachandran   contact map        clash detection
           |    DSSP-lite      SVM features       quality masks
           |         |              |
           v         v              v
         03/02 (this notebook)      |
                                    |
                    +---------------+
                    |               |
                   06/01           07/01
              (GNN adjacency)  (AF3 pLDDT)
```


In [ ]:
# ── Final summary: all quality metrics for one structure ─────────────────────

print('=' * 60)
print('STRUCTURE QUALITY REPORT — Simulated Protein')
print('=' * 60)

# Use the mixed-SS protein from Section 4
phi_mixed = np.concatenate([
    np.random.normal(-57, 8, 30),    # helix segment
    np.random.normal(-119, 10, 30),  # strand segment
    np.random.uniform(-180, 180, 40) # coil segment
])
psi_mixed = np.concatenate([
    np.random.normal(-47, 8, 30),
    np.random.normal(113, 10, 30),
    np.random.uniform(-180, 180, 40)
])

# 1. Ramachandran outliers
pct_out, n_out = ramachandran_outlier_pct(phi_mixed, psi_mixed)
rama_status = 'PASS' if pct_out < 0.5 else 'BORDERLINE' if pct_out < 2 else 'FAIL'

# 2. Secondary structure composition
ss_mixed = assign_ss_array(phi_mixed.tolist(), psi_mixed.tolist())
comp_mixed = ss_composition(ss_mixed)

# 3. Contact density
D_mix  = compute_distance_matrix(ca_coords)
C_mix  = compute_contact_map(D_mix)
n_contacts = C_mix.sum() // 2

# 4. B-factor summary
bf = simulate_bfactors(100)
pct_disordered = (bf > 60).sum() / 100 * 100

# 5. Clash score (from earlier)
clashes_report, cs_report = detect_clashes(grid_pts, clash_threshold=1.5)

print(f'\n1. Ramachandran outliers: {pct_out:.1f}% ({n_out}/{len(phi_mixed)})  [{rama_status}]')
print(f'   Threshold: <0.5% PASS, 0.5-2% BORDERLINE, >2% FAIL')

print(f'\n2. Secondary structure composition:')
for code in ['H', 'E', 'C']:
    print(f'   {code}: {comp_mixed[code]*100:.1f}%')

print(f'\n3. Contact map: {n_contacts} long-range contacts (seq sep >= 4, dist < 8A)')
print(f'   Contact density: {n_contacts / (100 * 99 / 2) * 100:.1f}%')

print(f'\n4. B-factor profile:')
print(f'   Mean B-factor: {bf.mean():.1f} A^2')
print(f'   Disordered residues (B > 60): {pct_disordered:.0f}%')

print(f'\n5. Clashscore: {cs_report:.1f} clashes/1000 atoms')
cs_status = 'PASS' if cs_report < 10 else 'BORDERLINE' if cs_report < 40 else 'FAIL'
print(f'   Status: [{cs_status}]')

print('\n' + '=' * 60)

---
## Summary

In this notebook you implemented from scratch, using only NumPy and Matplotlib:

1. **Dihedral angle calculation** — vector cross product method, `atan2` for signed angles. Showed why `arccos` is wrong (loses sign).
2. **Ramachandran plot** — scatter and density plots, identified alpha-helix and beta-strand clusters, drew allowed-region boundaries.
3. **Simplified DSSP** — rule-based secondary structure assignment from phi/psi angles, computed per-protein SS composition as an ML feature.
4. **Cα distance matrix and contact map** — broadcasting-based O(N²) computation, 8 Å threshold, sequential-neighbour masking, heatmap visualisation.
5. **Distance features for SVM** — upper-triangle flattening, StandardScaler, RBF-SVM achieving near-perfect classification of helix vs strand proteins.
6. **Cryo-EM concepts** — resolution, FSC, model-building pipeline, AF3 vs experimental structures.
7. **Structure quality metrics** — B-factor interpretation, Ramachandran outlier %, clash detection, MolProbity score components.
8. **Debug exercise** — identified three bugs in dihedral calculation: wrong bond direction, `arccos` vs `atan2`, missing `y` component.

**Next steps:**
- **Module 06/01:** Build a protein GNN where the contact map from this notebook defines the graph edges.
- **Module 07/01:** See how AlphaFold 3 uses pLDDT as a per-residue quality score, and how the training set was curated using structure quality filters.
- **Module 10/01:** Evaluate fine-tuned structure predictions using the quality metrics implemented here.


## Notebook Complete

**You can now:**
- Compute Ramachandran angles, secondary structure labels, and contact maps
- Normalise B-factors and interpret pLDDT confidence scores

**Where this fits:**
- Previous: 01_structure_analysis
- Next: 04_ml_bioinformatics/01_ml_for_omics — Module 04 — check the README

**Before moving on, check:**
- [ ] All code cells ran without errors
- [ ] You understand what each function does (read the docstrings)
- [ ] You can explain the key concept in one sentence without looking at notes